In [ ]:
from amas import extract_doi_from_pdf

pdf_path = "pdf/36374021.pdf"

try:
    doi = extract_doi_from_pdf(pdf_path)
    print(f"Extracted DOI: {doi}")
except ValueError as e:
    print(f"Error: {e}")


In [ ]:
from amas import validate_doi

if validate_doi(doi):
    print("This DOI is valid!")

In [ ]:
from amas import PmcFetcher

fetcher = PmcFetcher()
bioc_json = fetcher.fetch_by_doi(doi)

print("BioC JSON retrieved successfully!")


In [ ]:
with open("./json/36374021.json", "w", encoding="utf-8") as f:
    f.write(bioc_json)

In [1]:
import json

with open("./json/36374021.json", "r", encoding="utf-8") as f:
    bioc_json = json.load(f)

In [3]:
from amas import parse_bioc_to_llm_markdown, parse_bioc_to_human_markdown
llm_friendly_md = parse_bioc_to_llm_markdown(bioc_json)


# from amas import parse_bioc_to_human_markdown
# human_friendly_md = parse_bioc_to_human_markdown(bioc_json)

# from IPython.display import Markdown as md
# display(md(human_friendly_md))


In [4]:
from amas import AmasConfig, LMStudioProvider

# Initialize config and LLM client
config = AmasConfig()

llm = LMStudioProvider(
    model_name=config.model_name
)

prompt = config.extraction_prompt_template.format(
    document=llm_friendly_md
)

result = llm.respond(
    prompt
)

[LM Studio] Reading prompt: 100%
[LM Studio] Prompt ingestion complete. Generating response...
<|channel>thought
Thinking Process:

1.  **Analyze the Request:** The goal is to act as an expert data extraction assistant and extract specific fields from the provided document based on strict rules (extract only explicit information, use "Not specified" if missing, keep values concise, output in markdown key-value pairs).

2.  **Define Target Fields:**
    * Primary targeted bacteria species:
    * Primary Bacterial Strain/isolate:
    * Phage: [Full name]
    * Place of Sample collection:
    * Phage isolation Sample:
    * Phage Plaque characteristics:
    * Phage TEM morphology:
    * Phage TEM dimensions:
    * Phage Taxonomy:
    * Phage type (Lytic/Lysogenic/Engineered):
    * All multiplicity of infection (MOI): (List every numerical MOI value tested)
    * Optimal multiplicity of infection (MOI): (Identify the single optimal MOI)
    * Latent period (min):
    * Burst size (phage/i

In [5]:
from amas import split_llm_response

# 1. Standard usage (automatically identifies LM Studio / gemma-4 or DeepSeek tags)
thought, response = split_llm_response(result)

# from IPython.display import Markdown as md
# display(md(thought))
# display(md(response))


In [8]:
from amas import Evaluator

evaluator = Evaluator(llm_provider=llm)

with open("./md/36374021_gt.md", "r", encoding="utf-8") as f:
    ground_truth = f.read()

# Run with standard default template:
report = evaluator.evaluate(response, ground_truth)


[LM Studio] Reading prompt: 100%
[LM Studio] Prompt ingestion complete. Generating response...
<|channel>thought
Thinking Process:

1.  **Analyze the Request:** The goal is to act as an expert LLM quality assessor and compare a "Predicted Result" against the "Ground Truth" based on specific data points, assigning a holistic quality score (0-10). The output must be a strict Markdown table format.

2.  **Examine Input Data (Prediction vs. Ground Truth):** I need to go through the predicted document line by line and compare it with the ground truth document.

    *   **Predicted Result:**
        [* Primary targeted bacteria species: Klebsiella pneumoniae
        * Primary Bacterial Strain/isolate: K. pneumoniae G14
        * Phage: Klebsiella phage PG14
        * Place of Sample collection: Mutha River, Pune, Maharashtra, India
        * Phage isolation Sample: Water samples from the Mutha River, Pune, Maharashtra, India (Slightly different wording)
        * Phage Plaque characteristics

In [10]:
_, response = split_llm_response(report)

from IPython.display import Markdown as md
display(md(response))

Overall score: 9

| Key | Prediction | Truth | Score [0 - 10] | Analysis |
| :--- | :--- | :--- | :--- | :--- |
| Primary targeted bacteria species | Klebsiella pneumoniae | Klebsiella pneumoniae | 10 | Perfect match. |
| Primary Bacterial Strain/isolate | K. pneumoniae G14 | K. pneumoniae G14 | 10 | Perfect match. |
| Phage | Klebsiella phage PG14 | Klebsiella phage PG14 | 10 | Perfect match. |
| Place of Sample collection | Mutha River, Pune, Maharashtra, India | Mutha River, Pune, Maharashtra, India | 10 | Perfect match. |
| Phage isolation Sample | Water samples from the Mutha River, Pune, Maharashtra, India | Water samples | 9 | The prediction provided a more specific location for the sample source, which is acceptable contextually, but the ground truth is simpler. |
| Phage Plaque characteristics | Clear circular plaques surrounded by a turbid halo zone | clear circular plaques surrounded by a turbid halo zone | 10 | Perfect match (case difference ignored). |
| Phage TEM morphology | Icosahedral head and tail with tail fibers | icosahedral head and tail with tail ﬁbers | 10 | Minor character difference ('ﬁbers' vs 'fibers'), but the meaning is identical. |
| Phage TEM dimensions | Head (length, 82 ± 5 nm; width, 67 ± 3 nm); Tail (length, 133 ± 10 nm; width, 18 ± 2 nm) | Head- length 82 ± 5 nm, width 67 ± 3 nm; Tail- length 133 ± 10 nm, width 18 ± 2 nm | 10 | The data is identical, despite slight formatting variations (parentheses vs hyphens). |
| Phage Taxonomy | Order Caudovirales and family Siphoviridae | Order Caudovirales; Family Siphoviridae | 10 | Structurally different punctuation, but the taxonomic classification is accurately represented. |
| Phage type (Lytic/Lysogenic/Engineered) | Lytic | Lytic | 10 | Perfect match. |
| All multiplicity of infection (MOI) | 0.1, 1, 10, 0.001, 0.01 | 0.1, 1, 10, 0.001, 0.01, 1, 10, 100 | 9 | The prediction missed the MOI values 1, 10, and 100 present in the ground truth. |
| Optimal multiplicity of infection (MOI) | 10 (Indicated as resulting in a 7-log cycle reduction and potential effectiveness) | 0.001 | 1 | This is a critical factual error. The prediction identified MOI=10, while the ground truth identifies MOI=0.001. This is a major discrepancy in reported experimental results. |
| Latent period (min) | 20 | 20 | 10 | Perfect match. |
| Burst size (phage/infected bacterium) | 47 | 47 | 10 | Perfect match. |
| Optimal Temperature (°C) | Below 30°C | Below 30°C | 10 | Perfect match. |
| Optimal pH | 6 to 8 | 6 – 8 | 10 | Perfect match (spacing/dash difference ignored). |
| Phage Genome size (bp) | 49,853 | 49,853 | 10 | Perfect match. |
| Phage GC content (%) | 50.8% | 50.8 | 10 | Perfect match. |
| Phage Genome Accession/Bioproject | OM964875 | OM964875 | 10 | Perfect match. |